<a href="https://colab.research.google.com/github/ekomissarov/sutva-tsm/blob/main/tsm_events_aggregation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Документация к SQL-пайплайну подготовки panel-таблицы для switchback-эксперимента

## 1. Назначение

SQL-пайплайн формирует агрегированную panel-таблицу для анализа switchback A/B-теста в двустороннем marketplace, например ride-hailing сервисе.

Единица наблюдения в итоговой таблице:

\[
(region\_id,\ interval\_start)
\]

то есть один регион в одном экспериментальном временном окне.

В текущей конфигурации длина окна составляет 30 минут:

```sql
interval '30 minutes'
```

Для каждого такого окна рассчитываются:

- характеристики спроса;
- результаты матчинга;
- завершённые и отменённые поездки;
- GMV;
- показатели supply;
- среднее количество одновременно доступных водителей;
- нагрузка на доступный supply;
- treatment assignment;
- treatment предыдущего временного окна.

Все ride-level outcomes относятся к тому экспериментальному окну, в котором была создана исходная заявка `ride_requested`.

---

# 2. Входные таблицы

## 2.1. `ride_requested`

Таблица содержит события создания заявки на поездку.

Каждая строка соответствует событию создания одной поездки.

Пример:

| event_time | user_id | ride_id | region_id |
|---|---|---|---|
| 08:01:15 | U123 | R001 | South |
| 08:04:10 | U918 | R002 | South |

### Поля

| Поле | Описание |
|---|---|
| `event_time` | Время создания заявки |
| `user_id` | Идентификатор пользователя |
| `ride_id` | Идентификатор поездки |
| `region_id` | Регион, в котором была создана заявка |

`ride_id` используется как основной идентификатор для связывания заявки с последующими событиями поездки.

Перед дальнейшей обработкой таблица дедуплицируется по `ride_id`. Если существует несколько записей одного `ride_id`, используется самая ранняя:

```sql
row_number() over (
    partition by ride_id
    order by event_time asc
)
```

Это предотвращает размножение строк при последующих `JOIN`.

---

## 2.2. `match_created`

Таблица содержит события назначения водителя на поездку.

Пример:

| event_time | ride_id | driver_id |
|---|---|---|
| 08:02:14 | R001 | D041 |

### Поля

| Поле | Описание |
|---|---|
| `event_time` | Время создания матча |
| `ride_id` | Идентификатор поездки |
| `driver_id` | Идентификатор назначенного водителя |

Для каждой поездки используется **первый зарегистрированный match**.

Поэтому метрика времени ожидания имеет семантику:

\[
wait\_time =
first\_match\_time - request\_time
\]

Это время от создания заявки до первого назначения водителя.

Если в продукте возможны rematch-сценарии, последующие `match_created` для той же поездки в текущей версии пайплайна не используются.

---

## 2.3. `ride_completed`

Таблица содержит события завершения поездок.

Пример:

| event_time | ride_id | price |
|---|---|---:|
| 08:26:50 | R001 | 435 |

### Поля

| Поле | Описание |
|---|---|
| `event_time` | Время завершения поездки |
| `ride_id` | Идентификатор поездки |
| `price` | Стоимость завершённой поездки |

Для каждого `ride_id` используется первая зарегистрированная completion-запись.

Это защищает расчёт GMV от технических дублей, например при at-least-once доставке событий.

---

## 2.4. `ride_cancelled`

Таблица содержит события отмены поездок.

Пример:

| event_time | ride_id | reason |
|---|---|---|
| 08:05:44 | R002 | driver_timeout |

### Поля

| Поле | Описание |
|---|---|
| `event_time` | Время отмены |
| `ride_id` | Идентификатор поездки |
| `reason` | Причина отмены |

Для каждой поездки используется первая зарегистрированная cancellation-запись.

Отдельно рассчитывается количество отмен по причине:

```text
driver_timeout
```

---

## 2.5. `driver_online`

Несмотря на название, таблица содержит не только факт выхода водителя online, а **события изменения состояния водителя**.

Фактическая семантика таблицы ближе к `driver_status_changed`.

Пример:

| event_time | driver_id | region_id | status |
|---|---|---|---|
| 08:00:00 | D041 | South | available |
| 08:05:00 | D041 | South | busy |
| 08:27:00 | D041 | South | available |
| 08:45:00 | D041 | South | offline |

### Поля

| Поле | Описание |
|---|---|
| `event_time` | Время перехода в новое состояние |
| `driver_id` | Идентификатор водителя |
| `region_id` | Регион нахождения водителя |
| `status` | Новое состояние водителя |

Допустимые значения:

```text
available
busy
offline
```

### Семантика статусов

`available` означает, что водитель находится online и может принять новую заявку.

`busy` означает, что водитель находится online, но занят и в данный момент недоступен для нового матчинга.

`offline` означает, что водитель не является частью текущего supply.

Последовательность событий:

```text
08:00 available
08:05 busy
08:27 available
08:45 offline
```

преобразуется в интервалы состояния:

```text
[08:00, 08:05) available
[08:05, 08:27) busy
[08:27, 08:45) available
```

`offline`-сегменты в supply-агрегации не используются.

---

## 2.6. `switchback_assignments`

Таблица содержит экспериментальное назначение для каждого региона и временного окна.

Пример:

| interval_start | region_id | treatment |
|---|---|---:|
| 08:00 | South | 1 |
| 08:30 | South | 1 |
| 09:00 | South | 0 |
| 09:30 | South | 0 |

### Поля

| Поле | Описание |
|---|---|
| `interval_start` | Начало switchback-окна |
| `region_id` | Регион |
| `treatment` | Экспериментальное назначение |

В текущей схеме:

```text
treatment = 1 → treatment
treatment = 0 → control
```

Таблица должна содержать полную экспериментальную сетку `(region_id, interval_start)`.

Она одновременно используется как:

- источник treatment assignment;
- источник полного набора экспериментальных окон;
- основа для предотвращения пропусков при вычислении `LAG()`.

---

# 3. Параметры временного биннинга

В начале запроса задаются:

```sql
interval '30 minutes' as interval_len
```

и:

```sql
timestamp '2000-01-01 00:00:00' as bin_origin
```

`interval_len` задаёт продолжительность switchback-окна.

`bin_origin` задаёт общую точку отсчёта для функции:

```sql
date_bin(...)
```

Критически важно, чтобы тот же `bin_origin` использовался при построении `switchback_assignments`.

Иначе одинаковое событие может попасть в один временной бин в event pipeline и в другой — в assignment pipeline, после чего `JOIN` по `interval_start` станет некорректным.

---

# 4. Атрибуция поездок к switchback-окнам

Каждая заявка получает:

```sql
interval_start
```

на основании времени `ride_requested`.

Например:

```text
request = 08:25
```

при 30-минутных окнах относится к:

```text
interval_start = 08:00
```

Все дальнейшие outcomes этой поездки наследуют **тот же** `interval_start`.

Например:

```text
request     08:25
match       08:28
completion  08:47
```

все показатели поездки относятся к:

```text
08:00
```

Несмотря на то, что `completion` произошёл уже в следующем физическом временном окне.

Таким образом treatment определяется состоянием эксперимента **в момент создания заявки**:

\[
T_i =
T(region_i,\ request\_time_i)
\]

Это относится к таким метрикам, как:

- `matched`;
- `completed`;
- `gmv`;
- `cancelled`;
- `avg_wait_time_matched`.

---

# 5. Формирование driver-state сегментов

Исходные driver events сначала очищаются от полностью идентичных дублей:

```sql
select distinct
    driver_id,
    region_id,
    event_time,
    status
```

После этого удаляются последовательные повторения одинакового состояния.

Например:

```text
08:00 available
08:01 available
08:02 available
08:07 busy
```

преобразуются в:

```text
08:00 available
08:07 busy
```

Если водитель меняет регион, это также считается началом нового состояния, даже если `status` формально не изменился.

Например:

```text
08:00 South available
08:20 North available
```

создаёт два сегмента:

```text
South [08:00, 08:20) available
North [08:20, ...)
```

Конец каждого сегмента определяется следующим driver event через:

```sql
lead(event_time)
```

---

# 6. Расчёт supply

Для каждого экспериментального окна:

\[
[t,\ t+L)
\]

driver-state сегмент включается в расчёт, если существует непустое пересечение:

\[
state\_start < t+L
\]

и:

\[
state\_end > t
\]

Используются half-open intervals:

\[
[start,\ end)
\]

Это означает, например, что событие ровно в `08:30:00` уже относится к окну:

```text
[08:30, 09:00)
```

а не к:

```text
[08:00, 08:30)
```

Для каждого пересечения рассчитывается фактическое число минут, проведённых водителем в состоянии `available` или `busy`.

---

# 7. Выходная таблица

Финальный результат содержит **одну строку на каждый `(region_id, interval_start)`** из `switchback_assignments`.

Например:

| region_id | interval_start | treatment | requests | matched | completed | ... |
|---|---|---:|---:|---:|---:|---|
| South | 08:00 | 1 | 120 | 108 | 94 | ... |
| South | 08:30 | 1 | 134 | 119 | 101 | ... |
| South | 09:00 | 0 | 127 | 109 | 96 | ... |

Даже если внутри экспериментального окна нет ни одной заявки или поездки, строка сохраняется благодаря полной сетке `switchback_assignments`.

---

# 8. Описание выходных полей

## `region_id`

Регион, соответствующий данной строке panel-таблицы.

Единица экспериментальной рандомизации включает комбинацию региона и временного окна.

---

## `interval_start`

Начало switchback-окна.

Например:

```text
08:00
08:30
09:00
09:30
```

При `interval_len = 30 minutes` строка `08:00` соответствует интервалу:

\[
[08{:}00,\ 08{:}30)
\]

---

## `treatment`

Treatment assignment данного региона в данном временном окне.

```text
1 = treatment
0 = control
```

---

## `requests`

Количество уникальных поездок, созданных в данном регионе внутри данного окна:

\[
Requests_{r,t}
\]

Считается по `ride_requested`.

---

## `matched`

Количество заявок из данного окна, которые получили хотя бы один match.

Поскольку используется первый `match_created`, каждая поездка учитывается максимум один раз.

---

## `completed`

Количество заявок из данного request-окна, которые в дальнейшем завершились успешно.

Важно: поездка учитывается в окне **создания заявки**, а не в окне фактического завершения.

---

## `gmv`

Сумма `price` всех завершённых поездок, заявки на которые были созданы в данном окне:

\[
GMV_{r,t}
=
\sum_i price_i
\]

---

## `cancelled`

Количество заявок из данного окна, для которых была зарегистрирована отмена.

---

## `cancelled_driver_timeout`

Количество отмен, для которых:

```text
reason = 'driver_timeout'
```

Метрика может использоваться как дополнительный индикатор проблем со стороны supply или matching process.

---

# 9. Supply-поля

## `drivers_available`

Количество уникальных водителей, которые **хотя бы часть данного окна** находились в состоянии:

```text
available
```

Например, если:

```text
D1 available 08:00–08:05
D2 available 08:25–08:30
```

то:

```text
drivers_available = 2
```

Даже несмотря на то, что эти два водителя никогда не были `available` одновременно.

Поэтому поле **не является concurrent supply**.

---

## `drivers_busy`

Количество уникальных водителей, которые хотя бы часть окна находились в состоянии:

```text
busy
```

---

## `drivers_online_total`

Количество уникальных водителей, которые хотя бы часть окна находились либо:

```text
available
```

либо:

```text
busy
```

Один водитель учитывается максимум один раз, даже если внутри одного окна переходил между `available` и `busy`.

---

## `available_driver_minutes`

Суммарное количество минут, проведённых всеми водителями в состоянии `available` внутри данного окна.

Формально:

\[
AvailableDriverMinutes_{r,t}
=
\sum_d AvailableMinutes_{d,r,t}
\]

Например:

```text
D1: 30 available minutes
D2: 20 available minutes
D3: 10 available minutes
```

даёт:

\[
AvailableDriverMinutes = 60
\]

---

## `busy_driver_minutes`

Суммарное число минут, проведённых водителями в состоянии `busy`.

\[
BusyDriverMinutes_{r,t}
=
\sum_d BusyMinutes_{d,r,t}
\]

---

## `online_driver_minutes`

Суммарное online-время supply:

\[
OnlineDriverMinutes
=
AvailableDriverMinutes
+
BusyDriverMinutes
\]

В SQL:

```sql
available_driver_minutes
+
busy_driver_minutes
```

---

## `avg_available_drivers`

Среднее число одновременно доступных водителей в течение окна.

Рассчитывается как:

\[
AvgAvailableDrivers
=
\frac{AvailableDriverMinutes}
{WindowLengthMinutes}
\]

Для 30-минутного окна:

\[
AvgAvailableDrivers
=
\frac{AvailableDriverMinutes}{30}
\]

Например:

\[
\frac{60}{30}=2
\]

означает, что в среднем на протяжении данного 30-минутного окна одновременно были доступны два водителя.

Это одна из основных характеристик эффективного supply.

---

## `avg_online_drivers`

Среднее количество одновременно online-водителей независимо от их занятости:

\[
AvgOnlineDrivers
=
\frac{
AvailableDriverMinutes + BusyDriverMinutes
}{
WindowLengthMinutes
}
\]

Включает как `available`, так и `busy` supply.

---

# 10. Marketplace-метрики

## `load_available`

Характеризует нагрузку спроса на доступный supply:

\[
LoadAvailable
=
\frac{Requests}
{AvgAvailableDrivers}
\]

Например:

```text
requests = 120
avg_available_drivers = 20
```

тогда:

\[
LoadAvailable=6
\]

То есть на одну среднюю одновременно доступную единицу supply пришлось 6 заявок за экспериментальное окно.

Если:

```text
avg_available_drivers = 0
```

метрика возвращается как `NULL`.

---

## `match_rate`

Доля заявок, которые получили match:

\[
MatchRate
=
\frac{Matched}{Requests}
\]

Если в окне нет requests, возвращается `NULL`.

---

## `avg_wait_time_matched`

Среднее время от создания заявки до первого match:

\[
AvgWaitTimeMatched
=
E[
MatchTime - RequestTime
\mid Matched
]
\]

Измеряется в минутах.

Это **условная метрика только среди успешно matched заявок**.

Поэтому её необходимо интерпретировать совместно с `match_rate`.

Например treatment теоретически может привести к тому, что самые проблемные заявки вообще перестанут матчиться. Тогда `match_rate` снизится, но `avg_wait_time_matched` среди оставшихся заявок может уменьшиться.

---

# 11. `treat_lag1`

Поле:

```sql
lag(treatment) over (
    partition by region_id
    order by interval_start
)
```

содержит treatment assignment **предыдущего switchback-окна того же региона**.

Например:

| interval_start | treatment | treat_lag1 |
|---|---:|---:|
| 08:00 | 1 | NULL |
| 08:30 | 1 | 1 |
| 09:00 | 0 | 1 |
| 09:30 | 0 | 0 |

`treat_lag1` не относится к конкретной поездке.

Он означает:

> какой experimental condition действовал в данном регионе в непосредственно предыдущем switchback-окне.

Поле может использоваться для анализа carryover / lagged treatment effects.

---

# 12. Основные группы выходных метрик

Итоговую panel-таблицу удобно интерпретировать как комбинацию четырёх групп данных.

### Experiment assignment

```text
region_id
interval_start
treatment
treat_lag1
```

### Demand и ride outcomes

```text
requests
matched
completed
gmv
cancelled
cancelled_driver_timeout
```

### Supply

```text
drivers_available
drivers_busy
drivers_online_total
available_driver_minutes
busy_driver_minutes
online_driver_minutes
avg_available_drivers
avg_online_drivers
```

### Marketplace balance / matching

```text
load_available
match_rate
avg_wait_time_matched
```

Таким образом итоговая строка описывает одновременно experimental assignment, спрос, supply и результаты функционирования marketplace в конкретном регионе и конкретном switchback-окне.

In [ ]:
import pandas as pd
import numpy as np

# Magics
from helpers import (
    load_sql_magic,
    load_viz_magic
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)
load_viz_magic()          # %%viz   — open a DataFrame in PyGWalker (drag-and-drop charts)

# ============================================================
# 1. SWITCHBACK ASSIGNMENTS
# ============================================================

intervals = pd.date_range(
    start="2026-08-01 08:00:00",
    end="2026-08-01 11:30:00",
    freq="30min"
)

regions = ["South", "North"]

assignments = []

# South:
# T T C C T T C C
south_treatment = [1, 1, 0, 0, 1, 1, 0, 0]

# North:
# C C T T C C T T
north_treatment = [0, 0, 1, 1, 0, 0, 1, 1]

for region, treatment_pattern in [
    ("South", south_treatment),
    ("North", north_treatment),
]:
    for interval_start, treatment in zip(intervals, treatment_pattern):
        assignments.append(
            {
                "region_id": region,
                "interval_start": interval_start,
                "treatment": treatment,
            }
        )

switchback_assignments = pd.DataFrame(assignments)


# ============================================================
# 2. RIDE REQUESTED
# ============================================================

ride_requested = pd.DataFrame(
    [
        # ----------------------------------------------------
        # SOUTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:01:15",
            "user_id": "U001",
            "ride_id": "R001",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 08:07:00",
            "user_id": "U002",
            "ride_id": "R002",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 08:25:00",
            "user_id": "U003",
            "ride_id": "R003",
            "region_id": "South",
        },

        # технический дубль R003
        {
            "event_time": "2026-08-01 08:25:02",
            "user_id": "U003",
            "ride_id": "R003",
            "region_id": "South",
        },

        {
            "event_time": "2026-08-01 08:32:00",
            "user_id": "U004",
            "ride_id": "R004",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 08:48:00",
            "user_id": "U005",
            "ride_id": "R005",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 09:04:00",
            "user_id": "U006",
            "ride_id": "R006",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 09:22:00",
            "user_id": "U007",
            "ride_id": "R007",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 09:44:00",
            "user_id": "U008",
            "ride_id": "R008",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 10:03:00",
            "user_id": "U009",
            "ride_id": "R009",
            "region_id": "South",
        },
        {
            "event_time": "2026-08-01 10:41:00",
            "user_id": "U010",
            "ride_id": "R010",
            "region_id": "South",
        },

        # ----------------------------------------------------
        # NORTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:03:00",
            "user_id": "U101",
            "ride_id": "R101",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 08:17:00",
            "user_id": "U102",
            "ride_id": "R102",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 08:39:00",
            "user_id": "U103",
            "ride_id": "R103",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 09:10:00",
            "user_id": "U104",
            "ride_id": "R104",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 09:37:00",
            "user_id": "U105",
            "ride_id": "R105",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 10:12:00",
            "user_id": "U106",
            "ride_id": "R106",
            "region_id": "North",
        },
        {
            "event_time": "2026-08-01 11:02:00",
            "user_id": "U107",
            "ride_id": "R107",
            "region_id": "North",
        },
    ]
)

ride_requested["event_time"] = pd.to_datetime(
    ride_requested["event_time"]
)


# ============================================================
# 3. MATCH CREATED
# ============================================================

match_created = pd.DataFrame(
    [
        # ----------------------------------------------------
        # SOUTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:02:14",
            "ride_id": "R001",
            "driver_id": "D001",
        },

        {
            "event_time": "2026-08-01 08:09:00",
            "ride_id": "R002",
            "driver_id": "D002",
        },

        {
            "event_time": "2026-08-01 08:28:00",
            "ride_id": "R003",
            "driver_id": "D003",
        },

        # rematch:
        # SQL должен взять первый match R003
        {
            "event_time": "2026-08-01 08:31:00",
            "ride_id": "R003",
            "driver_id": "D004",
        },

        {
            "event_time": "2026-08-01 08:36:00",
            "ride_id": "R004",
            "driver_id": "D001",
        },

        # R005 специально НЕ имеет match

        {
            "event_time": "2026-08-01 09:08:00",
            "ride_id": "R006",
            "driver_id": "D002",
        },

        {
            "event_time": "2026-08-01 09:29:00",
            "ride_id": "R007",
            "driver_id": "D003",
        },

        {
            "event_time": "2026-08-01 09:47:00",
            "ride_id": "R008",
            "driver_id": "D001",
        },

        {
            "event_time": "2026-08-01 10:05:00",
            "ride_id": "R009",
            "driver_id": "D002",
        },

        # R010 без match

        # ----------------------------------------------------
        # NORTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:05:00",
            "ride_id": "R101",
            "driver_id": "D101",
        },
        {
            "event_time": "2026-08-01 08:20:00",
            "ride_id": "R102",
            "driver_id": "D102",
        },
        {
            "event_time": "2026-08-01 08:42:00",
            "ride_id": "R103",
            "driver_id": "D101",
        },
        {
            "event_time": "2026-08-01 09:14:00",
            "ride_id": "R104",
            "driver_id": "D103",
        },

        # R105 без match

        {
            "event_time": "2026-08-01 10:16:00",
            "ride_id": "R106",
            "driver_id": "D102",
        },
        {
            "event_time": "2026-08-01 11:08:00",
            "ride_id": "R107",
            "driver_id": "D103",
        },
    ]
)

match_created["event_time"] = pd.to_datetime(
    match_created["event_time"]
)


# ============================================================
# 4. RIDE COMPLETED
# ============================================================

ride_completed = pd.DataFrame(
    [
        # ----------------------------------------------------
        # SOUTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:26:50",
            "ride_id": "R001",
            "price": 435.0,
        },

        {
            "event_time": "2026-08-01 08:29:00",
            "ride_id": "R002",
            "price": 510.0,
        },

        # Request R003 был в 08:25,
        # completion уже в следующем switchback window.
        # В panel он должен остаться в interval_start = 08:00.
        {
            "event_time": "2026-08-01 08:47:00",
            "ride_id": "R003",
            "price": 620.0,
        },

        # технический дубль completion R003
        {
            "event_time": "2026-08-01 08:47:03",
            "ride_id": "R003",
            "price": 620.0,
        },

        {
            "event_time": "2026-08-01 08:58:00",
            "ride_id": "R004",
            "price": 390.0,
        },

        {
            "event_time": "2026-08-01 09:33:00",
            "ride_id": "R006",
            "price": 470.0,
        },

        {
            "event_time": "2026-08-01 09:55:00",
            "ride_id": "R007",
            "price": 580.0,
        },

        {
            "event_time": "2026-08-01 10:12:00",
            "ride_id": "R008",
            "price": 420.0,
        },

        {
            "event_time": "2026-08-01 10:35:00",
            "ride_id": "R009",
            "price": 530.0,
        },

        # ----------------------------------------------------
        # NORTH
        # ----------------------------------------------------
        {
            "event_time": "2026-08-01 08:30:00",
            "ride_id": "R101",
            "price": 440.0,
        },
        {
            "event_time": "2026-08-01 08:44:00",
            "ride_id": "R102",
            "price": 500.0,
        },
        {
            "event_time": "2026-08-01 09:06:00",
            "ride_id": "R103",
            "price": 650.0,
        },
        {
            "event_time": "2026-08-01 09:38:00",
            "ride_id": "R104",
            "price": 490.0,
        },
        {
            "event_time": "2026-08-01 10:43:00",
            "ride_id": "R106",
            "price": 560.0,
        },
        {
            "event_time": "2026-08-01 11:34:00",
            "ride_id": "R107",
            "price": 610.0,
        },
    ]
)

ride_completed["event_time"] = pd.to_datetime(
    ride_completed["event_time"]
)


# ============================================================
# 5. RIDE CANCELLED
# ============================================================

ride_cancelled = pd.DataFrame(
    [
        # SOUTH
        {
            "event_time": "2026-08-01 08:54:00",
            "ride_id": "R005",
            "reason": "driver_timeout",
        },

        # технический duplicate cancellation
        {
            "event_time": "2026-08-01 08:54:05",
            "ride_id": "R005",
            "reason": "driver_timeout",
        },

        {
            "event_time": "2026-08-01 10:49:00",
            "ride_id": "R010",
            "reason": "user_cancelled",
        },

        # NORTH
        {
            "event_time": "2026-08-01 09:44:00",
            "ride_id": "R105",
            "reason": "driver_timeout",
        },
    ]
)

ride_cancelled["event_time"] = pd.to_datetime(
    ride_cancelled["event_time"]
)


# ============================================================
# 6. DRIVER STATUS EVENTS
# ============================================================
#
# driver_online здесь фактически является driver_status_changed:
#
# available → водитель online и готов принять заказ
# busy      → водитель online, но занят
# offline   → водитель offline
#
# ============================================================

driver_online = pd.DataFrame(
    [
        # ====================================================
        # SOUTH — D001
        # ====================================================

        {
            "event_time": "2026-08-01 07:55:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "available",
        },

        # повтор available — должен быть удалён
        {
            "event_time": "2026-08-01 07:58:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "available",
        },

        {
            "event_time": "2026-08-01 08:02:14",
            "driver_id": "D001",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:26:50",
            "driver_id": "D001",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:36:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:58:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:47:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 10:12:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 10:30:00",
            "driver_id": "D001",
            "region_id": "South",
            "status": "offline",
        },

        # ====================================================
        # SOUTH — D002
        # ====================================================

        {
            "event_time": "2026-08-01 08:05:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:09:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:29:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:08:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 09:33:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 10:05:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 10:35:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 11:10:00",
            "driver_id": "D002",
            "region_id": "South",
            "status": "offline",
        },

        # ====================================================
        # SOUTH — D003
        # ====================================================

        {
            "event_time": "2026-08-01 08:20:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "available",
        },

        # exact duplicate
        {
            "event_time": "2026-08-01 08:20:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "available",
        },

        {
            "event_time": "2026-08-01 08:28:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:47:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:29:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 09:55:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 10:15:00",
            "driver_id": "D003",
            "region_id": "South",
            "status": "offline",
        },

        # ====================================================
        # D004 — смена региона без offline
        # ====================================================

        {
            "event_time": "2026-08-01 08:10:00",
            "driver_id": "D004",
            "region_id": "South",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:40:00",
            "driver_id": "D004",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:20:00",
            "driver_id": "D004",
            "region_id": "North",
            "status": "offline",
        },

        # ====================================================
        # NORTH — D101
        # ====================================================

        {
            "event_time": "2026-08-01 07:50:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:05:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:30:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:42:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 09:06:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:40:00",
            "driver_id": "D101",
            "region_id": "North",
            "status": "offline",
        },

        # ====================================================
        # NORTH — D102
        # ====================================================

        {
            "event_time": "2026-08-01 08:10:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 08:20:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 08:44:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 10:16:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 10:43:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 11:20:00",
            "driver_id": "D102",
            "region_id": "North",
            "status": "offline",
        },

        # ====================================================
        # NORTH — D103
        # ====================================================

        {
            "event_time": "2026-08-01 08:55:00",
            "driver_id": "D103",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 09:14:00",
            "driver_id": "D103",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 09:38:00",
            "driver_id": "D103",
            "region_id": "North",
            "status": "available",
        },
        {
            "event_time": "2026-08-01 11:08:00",
            "driver_id": "D103",
            "region_id": "North",
            "status": "busy",
        },
        {
            "event_time": "2026-08-01 11:34:00",
            "driver_id": "D103",
            "region_id": "North",
            "status": "available",
        },

        # Последний сегмент намеренно открыт.
        # SQL должен ограничить его analysis_end.
    ]
)

driver_online["event_time"] = pd.to_datetime(
    driver_online["event_time"]
)


# ============================================================
# 7. ПРОВЕРКА ТАБЛИЦ
# ============================================================

tables = {
    "ride_requested": ride_requested,
    "match_created": match_created,
    "ride_completed": ride_completed,
    "ride_cancelled": ride_cancelled,
    "driver_online": driver_online,
    "switchback_assignments": switchback_assignments,
}

for name, df in tables.items():
    print(f"\n{name}")
    print("-" * len(name))
    display(df.head(10))


ride_requested
--------------


,event_time,user_id,ride_id,region_id
0,2026-08-01 08:01:15,U001,R001,South
1,2026-08-01 08:07:00,U002,R002,South
2,2026-08-01 08:25:00,U003,R003,South
3,2026-08-01 08:25:02,U003,R003,South
4,2026-08-01 08:32:00,U004,R004,South
5,2026-08-01 08:48:00,U005,R005,South
6,2026-08-01 09:04:00,U006,R006,South
7,2026-08-01 09:22:00,U007,R007,South
8,2026-08-01 09:44:00,U008,R008,South
9,2026-08-01 10:03:00,U009,R009,South



match_created
-------------


,event_time,ride_id,driver_id
0,2026-08-01 08:02:14,R001,D001
1,2026-08-01 08:09:00,R002,D002
2,2026-08-01 08:28:00,R003,D003
3,2026-08-01 08:31:00,R003,D004
4,2026-08-01 08:36:00,R004,D001
5,2026-08-01 09:08:00,R006,D002
6,2026-08-01 09:29:00,R007,D003
7,2026-08-01 09:47:00,R008,D001
8,2026-08-01 10:05:00,R009,D002
9,2026-08-01 08:05:00,R101,D101



ride_completed
--------------


...

,event_time,ride_id,reason
0,2026-08-01 08:54:00,R005,driver_timeout
1,2026-08-01 08:54:05,R005,driver_timeout
2,2026-08-01 10:49:00,R010,user_cancelled
3,2026-08-01 09:44:00,R105,driver_timeout



driver_online
-------------


,event_time,driver_id,region_id,status
0,2026-08-01 07:55:00,D001,South,available
1,2026-08-01 07:58:00,D001,South,available
2,2026-08-01 08:02:14,D001,South,busy
3,2026-08-01 08:26:50,D001,South,available
4,2026-08-01 08:36:00,D001,South,busy
5,2026-08-01 08:58:00,D001,South,available
6,2026-08-01 09:47:00,D001,South,busy
7,2026-08-01 10:12:00,D001,South,available
8,2026-08-01 10:30:00,D001,South,offline
9,2026-08-01 08:05:00,D002,South,available



switchback_assignments
----------------------


,region_id,interval_start,treatment
0,South,2026-08-01 08:00:00,1
1,South,2026-08-01 08:30:00,1
2,South,2026-08-01 09:00:00,0
3,South,2026-08-01 09:30:00,0
4,South,2026-08-01 10:00:00,1
5,South,2026-08-01 10:30:00,1
6,South,2026-08-01 11:00:00,0
7,South,2026-08-01 11:30:00,0
8,North,2026-08-01 08:00:00,0
9,North,2026-08-01 08:30:00,0


In [ ]:
%%sql result <<

-- ============================================================
-- ПАРАМЕТРЫ
-- ============================================================

with params as (
    select
        interval '30 minutes' as interval_len,
        timestamp '2000-01-01 00:00:00' as bin_origin
        -- bin_origin должен совпадать с origin,
        -- использованным при построении switchback_assignments
),

-- ============================================================
-- 1. ГРАНИЦЫ ЭКСПЕРИМЕНТА
-- ============================================================

analysis_bounds as (
    select
        min(a.interval_start) as analysis_start,
        max(a.interval_start) + p.interval_len as analysis_end
    from switchback_assignments a
    cross join params p
    group by p.interval_len
),

-- ============================================================
-- 2. ДЕДУПЛИКАЦИЯ RIDE EVENTS
-- ============================================================

-- Одна каноническая заявка на ride_id.
--
-- Это важно сделать ДО последующих JOIN:
-- count(distinct ride_id) защитил бы requests,
-- но не защитил бы avg(wait_minutes) от размножения строк.
requests_deduped as (
    select
        ride_id,
        user_id,
        region_id,
        event_time
    from (
        select
            r.*,
            row_number() over (
                partition by ride_id
                order by event_time asc
            ) as rn
        from ride_requested r
    ) x
    where rn = 1
),

-- Первая completion-запись на ride_id.
-- Защита от retry / at-least-once delivery.
completions_deduped as (
    select
        ride_id,
        event_time,
        price
    from (
        select
            c.*,
            row_number() over (
                partition by ride_id
                order by event_time asc
            ) as rn
        from ride_completed c
    ) x
    where rn = 1
),

-- Первый match на ride_id.
--
-- Семантика:
-- wait time = время от request до первого назначения водителя.
matches_deduped as (
    select
        ride_id,
        driver_id,
        event_time
    from (
        select
            m.*,
            row_number() over (
                partition by ride_id
                order by event_time asc
            ) as rn
        from match_created m
    ) x
    where rn = 1
),

-- Первая cancellation-запись на ride_id.
cancellations_deduped as (
    select
        ride_id,
        event_time,
        reason
    from (
        select
            c.*,
            row_number() over (
                partition by ride_id
                order by event_time asc
            ) as rn
        from ride_cancelled c
    ) x
    where rn = 1
),

-- ============================================================
-- 3. БИННИНГ RIDE EVENTS
-- ============================================================

requests_binned as (
    select
        r.region_id,
        r.ride_id,
        r.user_id,
        r.event_time,
        p.bin_origin
        + floor(
            date_diff('second', p.bin_origin, r.event_time)
            / extract(epoch from p.interval_len)
          ) * p.interval_len as interval_start
    from requests_deduped r
    cross join params p
),

-- Все outcomes поездки приписываются окну REQUEST.
--
-- Например:
-- request    = 08:25
-- completion = 08:47
--
-- completion всё равно относится к interval_start = 08:00.
completions_binned as (
    select
        r.region_id,
        c.ride_id,
        c.price,
        r.interval_start
    from completions_deduped c
    join requests_binned r
      using (ride_id)
),

matches_binned as (
    select
        r.region_id,
        m.ride_id,
        m.driver_id,
        r.interval_start,

        extract(
            epoch from (m.event_time - r.event_time)
        ) / 60.0 as wait_minutes

    from matches_deduped m
    join requests_binned r
      using (ride_id)
),

cancellations_binned as (
    select
        r.region_id,
        c.ride_id,
        c.reason,
        r.interval_start
    from cancellations_deduped c
    join requests_binned r
      using (ride_id)
),

-- ============================================================
-- 4. DRIVER STATUS EVENTS
--
-- driver_online содержит события смены состояния:
--
-- status ∈ ('available', 'busy', 'offline')
--
-- Например:
--
-- 08:00 D041 South available
-- 08:05 D041 South busy
-- 08:27 D041 South available
-- 08:45 D041 South offline
-- ============================================================

-- Удаляем полностью идентичные event-дубли.
driver_events_exact_dedup as (
    select distinct
        driver_id,
        region_id,
        event_time,
        status
    from driver_online
),

-- Удаляем последовательные повторы одного состояния.
--
-- Например:
--
-- 08:00 available
-- 08:01 available
-- 08:02 available
-- 08:07 busy
--
-- превращается в:
--
-- 08:00 available
-- 08:07 busy
driver_events_dedup as (
    select
        driver_id,
        region_id,
        event_time,
        status
    from (
        select
            d.*,

            lag(status) over (
                partition by driver_id
                order by event_time
            ) as prev_status,

            lag(region_id) over (
                partition by driver_id
                order by event_time
            ) as prev_region

        from driver_events_exact_dedup d
    ) x
    where prev_status is null
       or prev_status <> status
       or prev_region <> region_id
),

-- ============================================================
-- 5. СЕССИОНИЗАЦИЯ ВОДИТЕЛЕЙ
-- ============================================================

driver_segments_raw as (
    select
        driver_id,
        region_id,
        status,
        event_time as state_start,

        lead(event_time) over (
            partition by driver_id
            order by event_time
        ) as state_end

    from driver_events_dedup
),

-- Ограничиваем сегменты границами анализируемого эксперимента.
--
-- Если последнее состояние водителя открыто на момент окончания
-- выгрузки, считаем, что оно продолжается до analysis_end.
--
-- Это корректно при предположении, что driver status log
-- полностью покрывает период эксперимента.
driver_segments as (
    select
        s.driver_id,
        s.region_id,
        s.status,

        greatest(
            s.state_start,
            b.analysis_start
        ) as state_start,

        least(
            coalesce(s.state_end, b.analysis_end),
            b.analysis_end
        ) as state_end

    from driver_segments_raw s
    cross join analysis_bounds b

    where s.state_start < b.analysis_end
      and (
            s.state_end is null
            or s.state_end > b.analysis_start
          )
),

-- Offline-сегменты для supply не нужны.
supply_segments as (
    select
        driver_id,
        region_id,
        status,
        state_start,
        state_end
    from driver_segments
    where status in ('available', 'busy')
      and state_end > state_start
),

-- ============================================================
-- 6. RIDE-LEVEL АГРЕГАЦИИ
-- ============================================================

agg_requests as (
    select
        region_id,
        interval_start,
        count(distinct ride_id) as requests
    from requests_binned
    group by
        region_id,
        interval_start
),

agg_completions as (
    select
        region_id,
        interval_start,
        count(distinct ride_id) as completed,
        sum(price) as gmv
    from completions_binned
    group by
        region_id,
        interval_start
),

agg_matches as (
    select
        region_id,
        interval_start,

        count(distinct ride_id) as matched,

        avg(wait_minutes) as avg_wait_time_matched

    from matches_binned
    group by
        region_id,
        interval_start
),

agg_cancellations as (
    select
        region_id,
        interval_start,

        count(distinct ride_id) as cancelled,

        count(distinct ride_id)
            filter (
                where reason = 'driver_timeout'
            ) as cancelled_driver_timeout

    from cancellations_binned
    group by
        region_id,
        interval_start
),

-- ============================================================
-- 7. ПОЛНАЯ СЕТКА ЭКСПЕРИМЕНТА
-- ============================================================

full_grid as (
    select distinct
        region_id,
        interval_start
    from switchback_assignments
),

-- ============================================================
-- 8. SUPPLY
--
-- Для каждого switchback-окна считаем пересечение
-- driver-state segments с:
--
-- [interval_start, interval_start + interval_len)
--
-- Используются half-open intervals.
-- ============================================================

agg_supply as (
    select
        g.region_id,
        g.interval_start,

        -- ----------------------------------------------------
        -- UNIQUE DRIVERS
        --
        -- Эти метрики означают:
        -- "водитель хотя бы часть окна был в данном состоянии".
        --
        -- Они НЕ являются concurrent supply.
        -- ----------------------------------------------------

        count(distinct s.driver_id)
            filter (
                where s.status = 'available'
            ) as drivers_available,

        count(distinct s.driver_id)
            filter (
                where s.status = 'busy'
            ) as drivers_busy,

        count(distinct s.driver_id)
            as drivers_online_total,

        -- ----------------------------------------------------
        -- DRIVER-MINUTES
        -- ----------------------------------------------------

        sum(
            extract(
                epoch from (
                    least(
                        s.state_end,
                        g.interval_start + p.interval_len
                    )
                    -
                    greatest(
                        s.state_start,
                        g.interval_start
                    )
                )
            ) / 60.0
        ) filter (
            where s.status = 'available'
        ) as available_driver_minutes,

        sum(
            extract(
                epoch from (
                    least(
                        s.state_end,
                        g.interval_start + p.interval_len
                    )
                    -
                    greatest(
                        s.state_start,
                        g.interval_start
                    )
                )
            ) / 60.0
        ) filter (
            where s.status = 'busy'
        ) as busy_driver_minutes

    from full_grid g
    cross join params p

    left join supply_segments s
      on s.region_id = g.region_id

     -- overlap двух half-open intervals
     and s.state_start < g.interval_start + p.interval_len
     and s.state_end > g.interval_start

    group by
        g.region_id,
        g.interval_start
),

-- ============================================================
-- 9. ФИНАЛЬНАЯ PANEL TABLE
-- ============================================================

panel as (
    select
        g.region_id,
        g.interval_start,
        a.treatment,

        -- ----------------------------------------------------
        -- DEMAND / RIDE OUTCOMES
        -- ----------------------------------------------------

        coalesce(rq.requests, 0)
            as requests,

        coalesce(mt.matched, 0)
            as matched,

        coalesce(cm.completed, 0)
            as completed,

        coalesce(cm.gmv, 0)
            as gmv,

        coalesce(cn.cancelled, 0)
            as cancelled,

        coalesce(cn.cancelled_driver_timeout, 0)
            as cancelled_driver_timeout,

        -- ----------------------------------------------------
        -- UNIQUE DRIVER METRICS
        -- ----------------------------------------------------

        coalesce(sp.drivers_available, 0)
            as drivers_available,

        coalesce(sp.drivers_busy, 0)
            as drivers_busy,

        coalesce(sp.drivers_online_total, 0)
            as drivers_online_total,

        -- ----------------------------------------------------
        -- DRIVER-MINUTES
        -- ----------------------------------------------------

        coalesce(sp.available_driver_minutes, 0)
            as available_driver_minutes,

        coalesce(sp.busy_driver_minutes, 0)
            as busy_driver_minutes,

        coalesce(sp.available_driver_minutes, 0)
        + coalesce(sp.busy_driver_minutes, 0)
            as online_driver_minutes,

        -- ----------------------------------------------------
        -- AVERAGE CONCURRENT SUPPLY
        --
        -- Например:
        --
        -- 60 available driver-minutes
        -- / 30 minutes
        -- = в среднем 2 available drivers одновременно.
        -- ----------------------------------------------------

        coalesce(
            sp.available_driver_minutes,
            0
        )
        /
        (
            extract(epoch from p.interval_len) / 60.0
        ) as avg_available_drivers,

        (
            coalesce(sp.available_driver_minutes, 0)
            +
            coalesce(sp.busy_driver_minutes, 0)
        )
        /
        (
            extract(epoch from p.interval_len) / 60.0
        ) as avg_online_drivers,

        -- ----------------------------------------------------
        -- LOAD
        --
        -- requests / среднее число одновременно available
        -- водителей в окне.
        -- ----------------------------------------------------

        case
            when coalesce(sp.available_driver_minutes, 0) > 0
            then
                coalesce(rq.requests, 0)::float
                /
                (
                    sp.available_driver_minutes
                    /
                    (
                        extract(epoch from p.interval_len)
                        / 60.0
                    )
                )
            else null
        end as load_available,

        -- ----------------------------------------------------
        -- MATCH RATE
        -- ----------------------------------------------------

        case
            when coalesce(rq.requests, 0) > 0
            then
                coalesce(mt.matched, 0)::float
                / rq.requests
            else null
        end as match_rate,

        -- Условная метрика:
        --
        -- E[wait_time | matched]
        --
        -- смотреть вместе с match_rate.
        mt.avg_wait_time_matched

    from full_grid g

    cross join params p

    join switchback_assignments a
      using (
          region_id,
          interval_start
      )

    left join agg_requests rq
      using (
          region_id,
          interval_start
      )

    left join agg_matches mt
      using (
          region_id,
          interval_start
      )

    left join agg_completions cm
      using (
          region_id,
          interval_start
      )

    left join agg_cancellations cn
      using (
          region_id,
          interval_start
      )

    left join agg_supply sp
      using (
          region_id,
          interval_start
      )
)

-- ============================================================
-- 10. TREATMENT + PREVIOUS WINDOW TREATMENT
-- ============================================================

select
    *,

    lag(treatment) over (
        partition by region_id
        order by interval_start
    ) as treat_lag1

from panel

order by
    region_id,
    interval_start;

,region_id,interval_start,treatment,requests,matched,completed,gmv,cancelled,cancelled_driver_timeout,drivers_available,...,drivers_online_total,available_driver_minutes,busy_driver_minutes,online_driver_minutes,avg_available_drivers,avg_online_drivers,load_available,match_rate,avg_wait_time_matched,treat_lag1
0,North,2026-08-01 08:00:00,0,2,2,2,940.0,0,0,2,...,2,15.0,35.0,50.0,0.500000,1.666667,4.000000,1.0,2.500000,<NA>
1,North,2026-08-01 08:30:00,0,1,1,1,650.0,0,0,4,...,4,53.0,32.0,85.0,1.766667,2.833333,0.566038,1.0,3.000000,0
2,North,2026-08-01 09:00:00,1,1,1,1,490.0,0,0,4,...,4,88.0,22.0,110.0,2.933333,3.666667,0.340909,1.0,4.000000,0
3,North,2026-08-01 09:30:00,1,1,0,0,0.0,1,1,3,...,3,62.0,8.0,70.0,2.066667,2.333333,0.483871,0.0,NaN,1
4,North,2026-08-01 10:00:00,0,1,1,1,560.0,0,0,2,...,2,46.0,14.0,60.0,1.533333,2.000000,0.652174,1.0,4.000000,1
5,North,2026-08-01 10:30:00,0,0,0,0,0.0,0,0,2,...,2,47.0,13.0,60.0,1.566667,2.000000,0.000000,NaN,NaN,0
6,North,2026-08-01 11:00:00,1,1,1,1,610.0,0,0,2,...,2,28.0,22.0,50.0,0.933333,1.666667,1.071429,1.0,6.000000,0
7,North,2026-08-01 11:30:00,1,0,0,0,0.0,0,0,1,...,1,26.0,4.0,30.0,0.866667,1.000000,0.000000,NaN,NaN,1
8,South,2026-08-01 08:00:00,1,3,3,3,1565.0,0,0,4,...,4,38.4,46.6,85.0,1.280000,2.833333,2.343750,1.0,1.994444,<NA>
9,South,2026-08-01 08:30:00,1,2,1,1,390.0,1,1,4,...,4,61.0,39.0,100.0,2.033333,3.333333,0.983607,0.5,4.000000,1


In [ ]:
result.columns

Index(['region_id', 'interval_start', 'treatment', 'requests', 'matched',
       'completed', 'gmv', 'cancelled', 'cancelled_driver_timeout',
       'drivers_available', 'drivers_busy', 'drivers_online_total',
       'available_driver_minutes', 'busy_driver_minutes',
       'online_driver_minutes', 'avg_available_drivers', 'avg_online_drivers',
       'load_available', 'match_rate', 'avg_wait_time_matched', 'treat_lag1'],
      dtype='str')